# 7. Export a query as PFB

`picsure::exportAsPFB(session, query, path)` runs a query server-side and writes the result as a [PFB](https://github.com/uc-cdis/pypfb) (Portable Format for Bioinformatics) file — Avro under the hood. The exported columns are the query's `includeConcepts`, so `buildQuery()` is the natural way to describe an export.

PFB export requires the Python `picsure[pfb]` optional dependency. If the Python env was provisioned without it, `exportAsPFB()` raises a `picsureError` pointing at the install hint.

In [ ]:
library(picsure)

In [ ]:
token_file <- "token.txt"
my_token <- readLines(token_file, warn = FALSE)[1]

In [ ]:
session <- picsure::connect(
  platform = picsure::Platform$BDC_AUTHORIZED,
  token    = my_token
)

## Find the concepts to filter on and export

In [ ]:
facets <- picsure::facets(session)
picsure::addFacet(facets, "dataset_id", "phs000007")

age_results <- picsure::searchDictionary(session, "age", facets = facets)
age5 <- age_results[age_results$display == "age5" & age_results$name == "phv00177938", ]

sex_results <- picsure::searchDictionary(session, "phv00253990", facets = facets)

## Build a query that filters and selects output columns

In [ ]:
age_filter <- picsure::buildClause(
  age5$conceptPath[[1]],
  type = picsure::PhenotypicFilterType$FILTER,
  min  = 30,
  max  = 40
)

query <- picsure::buildQuery(
  phenotypicFilter = age_filter,
  includeConcepts  = c(age5$conceptPath[[1]], sex_results$conceptPath[[1]])
)

## Confirm the cohort size, then write the PFB

`exportAsPFB()` returns the path invisibly, so you can capture it for downstream steps.

In [ ]:
picsure::runQuery(session, query, type = picsure::QueryType$COUNT)$value

In [ ]:
pfb_path <- picsure::exportAsPFB(session, query, "./cohort.avro")

file.info(pfb_path)[, c("size", "mtime")]

## Reading PFB back into R

The picsure package writes the file but doesn't take a position on how you read it back. PFB is plain Avro, so any Avro-capable R library works — `sparklyr::spark_read_avro()`, `arrow` with the Avro adapter, or shelling out to a Python helper inside the same reticulate env (since `fastavro` rides along with `picsure[pfb]`):

```r
fastavro <- reticulate::import("fastavro")
con      <- reticulate::import_builtins()$open(pfb_path, "rb")
records  <- reticulate::iterate(fastavro$reader(con))
length(records)
```

Pick whichever fits your downstream pipeline.